# Telco Customer Churn — Feature Engineering & Data Prep

Цель ноутбука — подготовить данные для моделирования churn:

- очистить и привести признаки к удобному формату;
- исключить ID, геопозицию и признаки с утечкой таргета;
- добавить бизнес-смысленные признаки (feature engineering);
- подготовить выборки `train`, `valid`, `test` для последующих моделей
  (Logistic Regression, Random Forest, CatBoost и др.).


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

RANDOM_STATE = 42


In [2]:
# загрузка исходного датасета
df_raw = pd.read_csv("../data/raw/telco_churn_raw.csv")

# копия для работы
df = df_raw.copy()

# приведение имён колонок к snake_case
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.head()


,customerid,count,country,state,city,zip_code,lat_long,latitude,longitude,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.96,-118.27,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.06,-118.31,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.05,-118.29,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.06,-118.32,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.04,-118.27,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


Дальнейшая обработка ведётся с колонками в формате `snake_case`
(как и в ноутбуке `01_eda_telco_churn.ipynb`), чтобы код оставался единообразным и читабельным.


In [3]:
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")

df["total_charges"].isna().sum(), df["total_charges"].dtype


(np.int64(11), dtype('float64'))

In [4]:
# простая стратегия: заменим NaN на 0 (клиент почти ничего не заплатил)
df["total_charges"] = df["total_charges"].fillna(0)


Признак `total_charges` явно приводится к числовому типу.
При возникновении некорректных значений (NaN) они заменяются на 0,
что соответствует клиентам с минимальной накопленной выручкой.


In [5]:
TARGET_COL = "churn_value"

id_cols = ["customerid"]
geo_cols = ["country", "state", "city", "lat_long", "zip_code", "latitude", "longitude"]
leakage_cols = ["churn_label", "churn_score", "churn_reason", "cltv"]
other_drop = ["count"]  # константа

drop_cols = id_cols + geo_cols + leakage_cols + other_drop

sorted(drop_cols)


['churn_label',
 'churn_reason',
 'churn_score',
 'city',
 'cltv',
 'count',
 'country',
 'customerid',
 'lat_long',
 'latitude',
 'longitude',
 'state',
 'zip_code']

In [6]:
set(drop_cols) - set(df.columns)


set()

In [7]:
df_model = df.drop(columns=drop_cols)

df_model.head()


,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn_value
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1


Из датасета удалены:

- идентификатор клиента (`customerid`);
- географические признаки (`country`, `state`, `city`, `lat_long`, `zip_code`, `latitude`, `longitude`);
- признаки с потенциальной утечкой таргета (`churn_label`, `churn_score`, `churn_reason`, `cltv`);
- служебная константа `count`.

Оставшиеся признаки содержат только ту информацию, которая доступна
на момент принятия решения о запуске retention-кампании.


In [8]:
df_fe = df_model.copy()

# 1. Новый клиент (<= 6 месяцев)
df_fe["is_new_customer"] = (df_fe["tenure_months"] <= 6).astype(int)

# 2. Долгосрочный контракт (год+)
df_fe["is_long_contract"] = df_fe["contract"].isin(["One year", "Two year"]).astype(int)

# 3. Есть ли интернет-услуга
df_fe["has_internet"] = (df_fe["internet_service"] != "No").astype(int)

# 4. Количество активных дополнительных услуг
addon_cols = [
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
]

# переведём Yes/No в 1/0 только для подсчёта
df_addons_binary = df_fe[addon_cols].apply(lambda col: col.eq("Yes").astype(int))
df_fe["num_addon_services"] = df_addons_binary.sum(axis=1)

# 5. Средняя месячная выручка
df_fe["avg_monthly_revenue"] = df_fe["total_charges"] / np.maximum(df_fe["tenure_months"], 1)

df_fe[[
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "is_new_customer",
    "is_long_contract",
    "has_internet",
    "num_addon_services",
    "avg_monthly_revenue",
]].head()


,tenure_months,monthly_charges,total_charges,is_new_customer,is_long_contract,has_internet,num_addon_services,avg_monthly_revenue
0,2,53.85,108.15,1,0,1,2,54.08
1,2,70.70,151.65,1,0,1,0,75.83
2,8,99.65,820.50,0,0,1,3,102.56
3,28,104.80,3046.05,0,0,1,4,108.79
4,49,103.70,5036.30,0,0,1,4,102.78


### Добавленные признаки

- `is_new_customer` — бинарный флаг новых клиентов (стаж ≤ 6 месяцев);
- `is_long_contract` — флаг долгосрочного контракта (1 или 2 года);
- `has_internet` — наличие интернет-услуги (любой тип);
- `num_addon_services` — количество активных дополнительных услуг
  (безопасность, бэкап, защита устройства, техподдержка, TV, фильмы);
- `avg_monthly_revenue` — средняя выручка в месяц (`total_charges / tenure_months`).

Эти признаки агрегируют результаты EDA в более компактную и бизнес-интерпретируемую форму,
что облегчает последующую интерпретацию моделей и построение retention-стратегий.


In [9]:
# целевая переменная
y = df_fe[TARGET_COL]

# признаки (всё кроме таргета)
X = df_fe.drop(columns=[TARGET_COL])

X.shape, y.shape


((7043, 24), (7043,))

In [10]:
num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

num_features, cat_features


C:\Users\badfa\AppData\Local\Temp\ipykernel_23920\3986848651.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include=["object"]).columns.tolist()


(['tenure_months',
  'monthly_charges',
  'total_charges',
  'is_new_customer',
  'is_long_contract',
  'has_internet',
  'num_addon_services',
  'avg_monthly_revenue'],
 ['gender',
  'senior_citizen',
  'partner',
  'dependents',
  'phone_service',
  'multiple_lines',
  'internet_service',
  'online_security',
  'online_backup',
  'device_protection',
  'tech_support',
  'streaming_tv',
  'streaming_movies',
  'contract',
  'paperless_billing',
  'payment_method'])

На этом этапе:

- `X` содержит все числовые и категориальные признаки, включая сгенерированные фичи;
- `y = churn_value` — бинарный таргет;
- `num_features` и `cat_features` будут использованы при построении пайплайнов
  (например, StandardScaler + OneHotEncoder для Logistic Regression
  и передаче списка категориальных признаков в CatBoost).


In [11]:
# сначала отделяем тестовую выборку
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

# затем делим train/valid внутри trainval
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,  # 0.25 от 0.8 = 0.2 от общего
    random_state=RANDOM_STATE,
    stratify=y_trainval,
)

X_train.shape, X_valid.shape, X_test.shape


((4225, 24), (1409, 24), (1409, 24))

In [12]:
y.value_counts(normalize=True), y_train.value_counts(normalize=True), y_valid.value_counts(normalize=True), y_test.value_counts(normalize=True)


(churn_value
 0   0.73
 1   0.27
 Name: proportion, dtype: float64,
 churn_value
 0   0.73
 1   0.27
 Name: proportion, dtype: float64,
 churn_value
 0   0.73
 1   0.27
 Name: proportion, dtype: float64,
 churn_value
 0   0.73
 1   0.27
 Name: proportion, dtype: float64)

Данные разделены на три части со стратификацией по `churn_value`:

- `train` ~60% — для обучения моделей;
- `valid` ~20% — для подбора гиперпараметров и выбора порогов;
- `test` ~20% — финальная независимая оценка качества.

Стратификация по таргету обеспечивает одинаковую долю churn во всех выборках.


In [13]:
import os

os.makedirs("../data/processed", exist_ok=True)

# полный датасет с фичами и таргетом
df_fe.to_csv("../data/processed/telco_churn_features_full.csv", index=False)

# отдельные выборки
train = X_train.copy()
train[TARGET_COL] = y_train

valid = X_valid.copy()
valid[TARGET_COL] = y_valid

test = X_test.copy()
test[TARGET_COL] = y_test

train.to_csv("../data/processed/train.csv", index=False)
valid.to_csv("../data/processed/valid.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)


Сохранены следующие файлы:

- `data/processed/telco_churn_features_full.csv` — полный набор признаков + таргет;
- `data/processed/train.csv` — обучающая выборка;
- `data/processed/valid.csv` — валидационная выборка;
- `data/processed/test.csv` — итоговая тестовая выборка.

Следующие ноутбуки (`03_modeling_baselines.ipynb`, `04_modeling_advanced_catboost.ipynb`)
будут работать уже с этими подготовленными данными.
